Script for telling the 3RRS where to go and record all the positions along the way:

In [10]:
import serial
import time
import re
import csv
from datetime import datetime
from pathlib import Path

PORT = '/dev/ttyACM0'
BAUD = 115200
READ_TIMEOUT_S = 0.2   # serial timeout for readline()
# How long with no serial traffic after last command before we assume we're done:
QUIET_GRACE_S = 5.0
# Max wait per point before we give up counting acks (still continue listening overall):
PER_POINT_ACK_TIMEOUT_S = 30.0

# Patterns that "acknowledge" a move; adjust to your firmware’s messages
ACK_PATTERNS = [
    r'\bok\b',
    r'\back\b',
    r'\breached\b',
    r'\bdone\b',
    r'\bcomplete(d)?\b',
    r'target\s*reached',
    r'at\s*target',
]
ACK_REGEX = re.compile('|'.join(f'(?:{p})' for p in ACK_PATTERNS), re.IGNORECASE)

LOG_FILE = Path('serial_log.csv')

def ts() -> str:
    # ISO timestamp with milliseconds
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]

def log_csv(writer, direction, text):
    writer.writerow([ts(), direction, text])

def send_with_log(ser, text, writer, flush=True):
    if not text.endswith('\n'):
        text += '\n'
    ser.write(text.encode())
    if flush:
        ser.flush()
    print(f"[{ts()}] >> {text.strip()}")
    log_csv(writer, 'TX', text.strip())

def listen_once(ser):
    line = ser.readline()  # bytes, blocked up to timeout
    if not line:
        return None
    try:
        decoded = line.decode(errors='ignore').strip()
    except Exception:
        decoded = repr(line)
    return decoded

def run_scan(points, delay_between_cmds=2.0, final_status_cmd='status'):
    # Open serial
    ser = serial.Serial(PORT, BAUD, timeout=READ_TIMEOUT_S)
    time.sleep(2.0)  # let the MCU reset if needed

    # Prepare CSV log
    new_file = not LOG_FILE.exists()
    f = LOG_FILE.open('a', newline='', encoding='utf-8')
    writer = csv.writer(f)
    if new_file:
        writer.writerow(['timestamp', 'dir', 'text'])  # header

    ack_count = 0
    total_points = len(points)
    last_io_time = time.time()
    per_point_start = time.time()

    print(f"[{ts()}] Starting scan of {total_points} points...")
    try:
        # Kick off a background listen while sending commands interleaved
        for (x, y, z) in points:
            # Send command
            cmd = f"circle_goto {x} {y} {z}"
            send_with_log(ser, cmd, writer)

            # After sending a command, listen for a bit so we don’t swamp the device
            send_time = time.time()
            per_point_start = send_time
            while time.time() - send_time < delay_between_cmds:
                line = listen_once(ser)
                if line is None:
                    # no data this tick
                    if (time.time() - per_point_start) > PER_POINT_ACK_TIMEOUT_S:
                        # stop waiting excessively for this point; move on
                        break
                    continue
                last_io_time = time.time()
                print(f"[{ts()}] << {line}")
                log_csv(writer, 'RX', line)
                if ACK_REGEX.search(line):
                    ack_count += 1

        # All commands sent — keep listening until:
        #  - we’ve seen ack_count == total_points, or
        #  - the port has been QUIET for QUIET_GRACE_S
        print(f"[{ts()}] All commands sent. Waiting for acks/quiet...")
        quiet_start = None
        while True:
            line = listen_once(ser)
            now = time.time()
            if line is None:
                # No line this tick
                if quiet_start is None:
                    quiet_start = now
                if ack_count >= total_points:
                    print(f"[{ts()}] Received {ack_count}/{total_points} acks. Stopping.")
                    break
                if (now - quiet_start) >= QUIET_GRACE_S:
                    print(f"[{ts()}] No serial activity for {QUIET_GRACE_S:.1f}s; assuming done.")
                    break
                continue

            # Got a line — reset quiet timer
            quiet_start = None
            last_io_time = now
            print(f"[{ts()}] << {line}")
            log_csv(writer, 'RX', line)
            if ACK_REGEX.search(line):
                ack_count += 1
                if ack_count >= total_points:
                    # Keep listening a short moment for any trailing status
                    tail_deadline = time.time() + 1.0
                    while time.time() < tail_deadline:
                        maybe = listen_once(ser)
                        if maybe:
                            print(f"[{ts()}] << {maybe}")
                            log_csv(writer, 'RX', maybe)
                    print(f"[{ts()}] Received {ack_count}/{total_points} acks. Stopping.")
                    break

        # Optionally ask for final state dump (adjust command to match your firmware)
        if final_status_cmd:
            send_with_log(ser, final_status_cmd, writer)
            # Read a short burst of responses
            end_deadline = time.time() + 2.0
            while time.time() < end_deadline:
                line = listen_once(ser)
                if not line:
                    continue
                print(f"[{ts()}] << {line}")
                log_csv(writer, 'RX', line)

    finally:
        ser.close()
        f.close()
        print(f"[{ts()}] Stopped. Serial closed. Log saved to {LOG_FILE.resolve()}")
        if ack_count < total_points:
            print(f"[{ts()}] Warning: only {ack_count}/{total_points} acknowledgements observed.")
        else:
            print(f"[{ts()}] All points acknowledged.")

# ---------------- Example points (your triangle scan) ----------------
sample_points = [
    (0, 0, 0),
    (100,   0, 400),
    (100, 400,   0),
    (  0,   0,   0),
    (100, 400,   0),
    (100,   0, 400),
    (  0,   0,   0),
    (100,   0, 400),
    (0, 0, 0)
]

if __name__ == "__main__":
    run_scan(sample_points, delay_between_cmds=2.0, final_status_cmd='status')


[2026-07-09 16:18:44.832] Starting scan of 9 points...
[2026-07-09 16:18:44.832] >> circle_goto 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-09 16:18:44.833] << P 0 0 0
[2026-07-0

SerialException: device reports readiness to read but returned no data (device disconnected or multiple access on port?)